In [ ]:
import os
import cv2
import uuid
from PIL import Image
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, ScalarQuantizationConfig, ScalarType, ScalarQuantization
import matplotlib.pyplot as plt

from playground.torch.patches import get_images

client = QdrantClient(location="http://localhost:6333", port=None, grpc_port=None, timeout=600) # Connect to existing Qdrant instance
client.create_collection(
    collection_name="sift",
    vectors_config=VectorParams(size=128, distance=Distance.MANHATTAN)
)

In [ ]:
artwork_images = get_images(
    source_pattern=os.path.join("..", "data", "artwork", "**"),
    skip=None
)

sift = cv2.SIFT_create()

i = 0
for file, q in artwork_images:
    # find and draw the keypoints
    keypoints, X = sift.detectAndCompute(q ,None)

    if keypoints is None or X is None:
        continue

    ids = [i + n for n in range(len(keypoints))]

    client.upsert(
        collection_name="sift",
        points=[
            PointStruct(
                    id=idx,
                    vector=Xi.tolist(),
                    payload={"file": file, "point": kp.pt}
            )
            for idx, kp, Xi in zip(ids, keypoints, X)
            ]
    )

    i = max(ids) + 1


Query SIFT collection

In [ ]:
test_images = get_images(
    source_pattern=os.path.join("..", "data", "query", "**"),
    skip=None
)

In [ ]:
for file, q in test_images:
    # find and draw the keypoints
    kp, des = sift.detectAndCompute(q ,None)
    
    img2 = cv2.drawKeypoints(q, kp, None, color=(255,0,0))

    print(des.shape)
    f, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(img2)
    break

In [ ]:
results = []

for Xi in tqdm(des):
    result = client.query_points(
        collection_name="sift",
        query=Xi,
        with_payload=True,
        limit=20,
        timeout=60
    )
    results.append(result)

In [ ]:
import json
P = []

for i, result in enumerate(results):
    Pi = pd.DataFrame(json.loads(result.json())["points"])
    Pi["point"] = i
    Pi["candidate"] = range(Pi.shape[0])
    P.append(Pi)
P = pd.concat(P, axis=0)

In [ ]:
P["payload"] = P["payload"].apply(lambda x: x["file"])

In [ ]:
P["payload"].value_counts()